# **0. Importación de Librerías**

In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import joblib
nltk.download('stopwords')
nltk.download('punkt')
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem.snowball import SnowballStemmer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
import time

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/vscode/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# **1. Exploración Inicial**

In [2]:
# URL
DATA_URL = "https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv"

# Carga el dataset
try:
    df = pd.read_csv(DATA_URL)
    print("Dataset cargado.")
except Exception as e:
    print(f"Error al cargar el dataset: {e}")

Dataset cargado.


## *1.1. Información inicial*

In [3]:
print("\n1. Vista Preliminar")
print(df.head())
print("\n2. Información General")
print(df.info())
print("\n3. Conteo de Clases (Columna 'is_spam')")
print(df['is_spam'].value_counts())


1. Vista Preliminar
                                                 url  is_spam
0  https://briefingday.us8.list-manage.com/unsubs...     True
1                             https://www.hvper.com/     True
2                 https://briefingday.com/m/v4n3i4f3     True
3   https://briefingday.com/n/20200618/m#commentform    False
4                        https://briefingday.com/fan     True

2. Información General
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2999 entries, 0 to 2998
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   url      2999 non-null   object
 1   is_spam  2999 non-null   bool  
dtypes: bool(1), object(1)
memory usage: 26.5+ KB
None

3. Conteo de Clases (Columna 'is_spam')
is_spam
False    2303
True      696
Name: count, dtype: int64


# **2. Preprocesamiento de Enlaces y Vectorización**

In [4]:
# Define el tokenizador
stemmer = SnowballStemmer("english")

# Son Stopwords específicas para URLs
url_stopwords = set(nltk.corpus.stopwords.words('english') + 
                    ['http', 'https', 'www', 'com', 'org', 'net', 'html', 'php', 'aspx', 'asp', 'htm', 'co', 'us', 'in', 'm'])

## *2.1. Segmentación y Limpieza*

In [14]:
def custom_tokenizer(url):

    # Separa la URL por cualquier caracter que no sea una letra o un número
    tokens = re.split(r'[^a-zA-Z0-9]', url)
    
    # Limpia y aplica Stemming
    cleaned_tokens = []
    for token in tokens:
        token = token.lower().strip()
        if len(token) > 1 and token not in url_stopwords:
        
            cleaned_tokens.append(stemmer.stem(token))
            
    return cleaned_tokens

## *2.2. Train y Test*

In [15]:
X = df['url']
y = df['is_spam']

# Train y Test (stratify=y sirve para asegurar la proporción de spam/no-spam)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

## *2.3. Vectorización y Entrenamiento en X_Train*

In [21]:
vectorizer = TfidfVectorizer(tokenizer=custom_tokenizer)

# Entrena el vectorizador
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("\nResultado de la Vectorización:")
print ("")
print(f"- X_train_vec shape: {X_train_vec.shape}")
print(f"- X_test_vec shape: {X_test_vec.shape}")
print(f"- Total de características (tokens únicos): {X_train_vec.shape[1]}")

# Guarda el vectorizador
joblib.dump(vectorizer, 'tfidf_vectorizer_url_spam.pkl')
print ("")
print("Vectorizador guardado.")


Resultado de la Vectorización:

- X_train_vec shape: (2099, 4738)
- X_test_vec shape: (900, 4738)
- Total de características (tokens únicos): 4738

Vectorizador guardado.


# **3. Entrenamiento y Análisis del modelo SVM**

In [22]:
# Construye el modelo
svm_model = SVC(random_state=42)

# Entrena el modelo con los datos vectorizados
svm_model.fit(X_train_vec, y_train)

# Predicción sobre el conjunto de prueba
y_pred = svm_model.predict(X_test_vec)

# Evaluación y Análisis de resultados
print("\nEvaluación del SVM (Parámetros por Defecto)")
accuracy = accuracy_score(y_test, y_pred)
print(f"- Precisión General (Accuracy): {accuracy:.4f}")

print("\n- Matriz de Confusión:\n", confusion_matrix(y_test, y_pred))

# Reporte de clasificación: métricas clave (Precisión, Recall, F1-Score)
print("\n- Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Spam', 'Spam']))


Evaluación del SVM (Parámetros por Defecto)
- Precisión General (Accuracy): 0.9533

- Matriz de Confusión:
 [[680  11]
 [ 31 178]]

- Reporte de Clasificación:
              precision    recall  f1-score   support

     No Spam       0.96      0.98      0.97       691
        Spam       0.94      0.85      0.89       209

    accuracy                           0.95       900
   macro avg       0.95      0.92      0.93       900
weighted avg       0.95      0.95      0.95       900



# **4. Optimización del modelo con GridSearch**

In [23]:

param_grid = {
    'C': [0.1, 1, 10], 
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 0.1, 1] 
}

# Configuración de GridSearchCV
# Usamos 'f1' ya que, como hemos visto antes, las clases están desbalanceadas.
grid_search = GridSearchCV(
    estimator=SVC(random_state=42), 
    param_grid=param_grid, 
    scoring='f1', 
    cv=3, 
    verbose=1,
    n_jobs=-1 
)

## *4.1. Búsqueda en datos de entrenamiento y Obtención de mejores resultados* 

In [ ]:
grid_search.fit(X_train_vec, y_train)

print("\nResultados de la Optimización")
print(f"- Mejores Parámetros: {grid_search.best_params_}")
print(f"- Mejor Puntuación F1 (Cross-Validation): {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 18 candidates, totalling 54 fits

Resultados de la Optimización
- Mejores Parámetros: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
- Mejor Puntuación F1 (Cross-Validation): 0.9204


## *4.2. Evaluación del mejor modelo y Reporte de Clasificación*

In [26]:
best_svm = grid_search.best_estimator_
y_pred_tuned = best_svm.predict(X_test_vec)

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"\n- Precisión General (Accuracy) del Modelo Optimizado: {accuracy_tuned:.4f}")

print("\n- Reporte de Clasificación (Optimizado):")
print(classification_report(y_test, y_pred_tuned, target_names=['No Spam', 'Spam']))


- Precisión General (Accuracy) del Modelo Optimizado: 0.9500

- Reporte de Clasificación (Optimizado):
              precision    recall  f1-score   support

     No Spam       0.97      0.97      0.97       691
        Spam       0.89      0.89      0.89       209

    accuracy                           0.95       900
   macro avg       0.93      0.93      0.93       900
weighted avg       0.95      0.95      0.95       900



# **5. Guardado del modelo**

In [27]:
best_svm = grid_search.best_estimator_

joblib.dump(best_svm, 'svm_url_spam_classifier.pkl')

print("Modelo SVM optimizado guardado como 'svm_url_spam_classifier.pkl'.")

Modelo SVM optimizado guardado como 'svm_url_spam_classifier.pkl'.
